In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style("whitegrid")
sns.set_palette("Set2")
plt.rcParams['figure.figsize'] = (14, 6)

# Create figures directory
import os
os.makedirs('/Users/gihbeom/Desktop/Data Science Preparation/fqhc-chronic-disease-analysis/figures', exist_ok=True)

# ============================================================
# LOAD PROCESSED DATA
# ============================================================

df = pd.read_pickle('/Users/gihbeom/Desktop/Data Science Preparation/fqhc-chronic-disease-analysis/data/processed/hc2024_processed.pkl')

print("="*80)
print("EXPLORATORY DATA ANALYSIS — NAMCS HC 2024")
print("="*80)
print(f"Dataset: {len(df):,} visits from 107 health centers\n")

# ============================================================
# VISUALIZATION 1: Chronic Disease Prevalence
# ============================================================

print("Creating Visualization 1: Chronic Disease Prevalence")

chronic_conditions = ['diabetes', 'hypertension', 'mental_health', 'obesity', 
                     'asthma_copd', 'chd', 'ckd', 'substance_use']

chronic_prev = df[chronic_conditions].sum().sort_values(ascending=True)
chronic_pct = (chronic_prev / len(df) * 100)

fig, ax = plt.subplots(figsize=(12, 6))

colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(chronic_prev)))
bars = ax.barh(range(len(chronic_prev)), chronic_pct, color=colors, alpha=0.8, edgecolor='black')

# Add value labels
for i, (count, pct) in enumerate(zip(chronic_prev, chronic_pct)):
    ax.text(pct + 0.3, i, f'{count:,} ({pct:.1f}%)', 
            va='center', fontsize=10, fontweight='bold')

# Labels and formatting
condition_labels = {
    'diabetes': 'Diabetes',
    'hypertension': 'Hypertension',
    'mental_health': 'Mental Health Disorders',
    'obesity': 'Obesity',
    'asthma_copd': 'Asthma/COPD',
    'chd': 'Coronary Heart Disease',
    'ckd': 'Chronic Kidney Disease',
    'substance_use': 'Substance Use Disorders'
}

ax.set_yticks(range(len(chronic_prev)))
ax.set_yticklabels([condition_labels[c] for c in chronic_prev.index], fontsize=11)
ax.set_xlabel('Percentage of Visits (%)', fontsize=12, fontweight='bold')
ax.set_title('Chronic Disease Prevalence in FQHC Visits (2024)', 
             fontsize=14, fontweight='bold', pad=20)
ax.grid(axis='x', alpha=0.3)
ax.set_xlim(0, max(chronic_pct) + 3)

plt.tight_layout()
plt.savefig('/Users/gihbeom/Desktop/Data Science Preparation/fqhc-chronic-disease-analysis/figures/01_chronic_disease_prevalence.png', 
            dpi=300, bbox_inches='tight')
print("✓ Saved: figures/01_chronic_disease_prevalence.png")
plt.close()

# ============================================================
# VISUALIZATION 2: Health Equity Analysis by Race/Ethnicity
# ============================================================

print("Creating Visualization 2: Health Equity Disparities")

# Calculate prevalence by race/ethnicity for top 3 conditions
top_conditions = ['diabetes', 'hypertension', 'mental_health']

equity_data = []
for condition in top_conditions:
    for race in ['White', 'Black', 'Hispanic', 'Other']:
        subset = df[df['race_simple'] == race]
        if len(subset) > 0:
            prev = subset[condition].mean() * 100
            equity_data.append({
                'Condition': condition_labels[condition],
                'Race/Ethnicity': race,
                'Prevalence': prev
            })

equity_df = pd.DataFrame(equity_data)

fig, ax = plt.subplots(figsize=(14, 6))

# Grouped bar chart
x = np.arange(len(top_conditions))
width = 0.2
races = ['White', 'Black', 'Hispanic', 'Other']
colors_race = {'White': '#3498db', 'Black': '#e74c3c', 'Hispanic': '#2ecc71', 'Other': '#f39c12'}

for i, race in enumerate(races):
    race_data = equity_df[equity_df['Race/Ethnicity'] == race]
    values = [race_data[race_data['Condition'] == condition_labels[c]]['Prevalence'].values[0] 
              if len(race_data[race_data['Condition'] == condition_labels[c]]) > 0 else 0
              for c in top_conditions]
    
    ax.bar(x + i*width, values, width, label=race, 
           color=colors_race[race], alpha=0.8, edgecolor='black')

ax.set_xlabel('Chronic Condition', fontsize=12, fontweight='bold')
ax.set_ylabel('Prevalence (%)', fontsize=12, fontweight='bold')
ax.set_title('Chronic Disease Disparities by Race/Ethnicity at FQHCs (2024)', 
             fontsize=14, fontweight='bold', pad=20)
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels([condition_labels[c] for c in top_conditions], fontsize=11)
ax.legend(title='Race/Ethnicity', fontsize=10, title_fontsize=11)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('/Users/gihbeom/Desktop/Data Science Preparation/fqhc-chronic-disease-analysis/figures/02_health_equity_disparities.png', 
            dpi=300, bbox_inches='tight')
print("✓ Saved: figures/02_health_equity_disparities.png")
plt.close()

# ============================================================
# VISUALIZATION 3: Visit Complexity Distribution
# ============================================================

print("Creating Visualization 3: Visit Complexity")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Number of diagnoses distribution
ax1 = axes[0]
diag_dist = df['num_diagnoses'].value_counts().sort_index()
diag_dist_truncated = diag_dist[diag_dist.index <= 15]  # Truncate at 15 for readability

ax1.bar(diag_dist_truncated.index, diag_dist_truncated.values, 
        color='steelblue', alpha=0.7, edgecolor='black')
ax1.set_xlabel('Number of Diagnoses per Visit', fontsize=11, fontweight='bold')
ax1.set_ylabel('Number of Visits', fontsize=11, fontweight='bold')
ax1.set_title('Distribution of Diagnoses per Visit', fontsize=12, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)

# Add mean line
mean_diag = df['num_diagnoses'].mean()
ax1.axvline(mean_diag, color='red', linestyle='--', linewidth=2, 
            label=f'Mean: {mean_diag:.1f}')
ax1.legend()

# Plot 2: Multimorbidity by age group
ax2 = axes[1]
multimorbidity_by_age = df.groupby('age_group_detailed')['multimorbidity'].mean() * 100

colors_age = plt.cm.viridis(np.linspace(0.2, 0.9, len(multimorbidity_by_age)))
bars = ax2.bar(range(len(multimorbidity_by_age)), multimorbidity_by_age.values, 
               color=colors_age, alpha=0.8, edgecolor='black')

ax2.set_xticks(range(len(multimorbidity_by_age)))
ax2.set_xticklabels(multimorbidity_by_age.index, rotation=45, ha='right')
ax2.set_xlabel('Age Group', fontsize=11, fontweight='bold')
ax2.set_ylabel('Multimorbidity Rate (%)', fontsize=11, fontweight='bold')
ax2.set_title('Multimorbidity (2+ Chronic Conditions) by Age', fontsize=12, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

# Add value labels
for i, v in enumerate(multimorbidity_by_age.values):
    ax2.text(i, v + 0.5, f'{v:.1f}%', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('/Users/gihbeom/Desktop/Data Science Preparation/fqhc-chronic-disease-analysis/figures/03_visit_complexity.png', 
            dpi=300, bbox_inches='tight')
print("✓ Saved: figures/03_visit_complexity.png")
plt.close()

# ============================================================
# VISUALIZATION 4: Seasonal Patterns
# ============================================================

print("Creating Visualization 4: Seasonal Utilization Patterns")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Monthly visit volume
ax1 = axes[0, 0]
monthly_visits = df.groupby('MONTH').size()
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

ax1.plot(monthly_visits.index, monthly_visits.values, 
         marker='o', linewidth=2, markersize=8, color='#2c3e50')
ax1.fill_between(monthly_visits.index, monthly_visits.values, alpha=0.3, color='#3498db')
ax1.set_xticks(range(1, 13))
ax1.set_xticklabels(month_names, rotation=45)
ax1.set_xlabel('Month', fontsize=11, fontweight='bold')
ax1.set_ylabel('Number of Visits', fontsize=11, fontweight='bold')
ax1.set_title('Monthly Visit Volume', fontsize=12, fontweight='bold')
ax1.grid(True, alpha=0.3)

# Plot 2: Seasonal chronic disease prevalence
ax2 = axes[0, 1]
seasonal_chronic = df.groupby('season')['any_chronic'].mean() * 100
season_order = ['Winter', 'Spring', 'Summer', 'Fall']
seasonal_chronic = seasonal_chronic.reindex(season_order)

colors_season = {'Winter': '#3498db', 'Spring': '#2ecc71', 
                 'Summer': '#f39c12', 'Fall': '#e74c3c'}
bars = ax2.bar(range(len(seasonal_chronic)), seasonal_chronic.values,
               color=[colors_season[s] for s in season_order], 
               alpha=0.8, edgecolor='black')

ax2.set_xticks(range(len(seasonal_chronic)))
ax2.set_xticklabels(season_order)
ax2.set_xlabel('Season', fontsize=11, fontweight='bold')
ax2.set_ylabel('Chronic Disease Prevalence (%)', fontsize=11, fontweight='bold')
ax2.set_title('Seasonal Chronic Disease Patterns', fontsize=12, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

for i, v in enumerate(seasonal_chronic.values):
    ax2.text(i, v + 0.3, f'{v:.1f}%', ha='center', fontsize=10, fontweight='bold')

# Plot 3: Day of week patterns
ax3 = axes[1, 0]
dow_dist = df.groupby('DAY').size()
day_names = ['Sun', 'Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat']

colors_dow = ['#e74c3c' if d in [1, 7] else '#3498db' for d in dow_dist.index]
ax3.bar(range(len(dow_dist)), dow_dist.values, 
        color=colors_dow, alpha=0.8, edgecolor='black')

ax3.set_xticks(range(len(dow_dist)))
ax3.set_xticklabels(day_names)
ax3.set_xlabel('Day of Week', fontsize=11, fontweight='bold')
ax3.set_ylabel('Number of Visits', fontsize=11, fontweight='bold')
ax3.set_title('Visit Volume by Day of Week', fontsize=12, fontweight='bold')
ax3.grid(axis='y', alpha=0.3)

# Plot 4: Preventive care by season
ax4 = axes[1, 1]
seasonal_preventive = df.groupby('season')['preventive_visit'].mean() * 100
seasonal_preventive = seasonal_preventive.reindex(season_order)

ax4.bar(range(len(seasonal_preventive)), seasonal_preventive.values,
        color=[colors_season[s] for s in season_order], 
        alpha=0.8, edgecolor='black')

ax4.set_xticks(range(len(seasonal_preventive)))
ax4.set_xticklabels(season_order)
ax4.set_xlabel('Season', fontsize=11, fontweight='bold')
ax4.set_ylabel('Preventive Visit Rate (%)', fontsize=11, fontweight='bold')
ax4.set_title('Preventive Care Utilization by Season', fontsize=12, fontweight='bold')
ax4.grid(axis='y', alpha=0.3)

for i, v in enumerate(seasonal_preventive.values):
    ax4.text(i, v + 0.15, f'{v:.1f}%', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('/Users/gihbeom/Desktop/Data Science Preparation/fqhc-chronic-disease-analysis/figures/04_seasonal_patterns.png', 
            dpi=300, bbox_inches='tight')
print("✓ Saved: figures/04_seasonal_patterns.png")
plt.close()

# ============================================================
# SUMMARY STATISTICS TABLE
# ============================================================

print("\n" + "="*80)
print("GENERATING SUMMARY STATISTICS")
print("="*80)

summary_stats = pd.DataFrame({
    'Metric': [
        'Total Visits',
        'Health Centers',
        'Weighted National Visits',
        'Average Age (years)',
        'Female (%)',
        'Hispanic (%)',
        'Black (%)',
        'Any Chronic Disease (%)',
        'Diabetes (%)',
        'Hypertension (%)',
        'Mental Health (%)',
        'Multimorbidity (%)',
        'High Complexity Visits (%)',
        'Average Diagnoses per Visit',
        'Preventive Care Visits (%)',
        'Weekend Visits (%)'
    ],
    'Value': [
        f"{len(df):,}",
        "107",
        "123,817,677",
        f"{df['AGE'].mean():.1f}",
        f"{(df['sex_binary']==1).sum() / len(df) * 100:.1f}%",
        f"{(df['race_simple']=='Hispanic').sum() / len(df) * 100:.1f}%",
        f"{(df['race_simple']=='Black').sum() / len(df) * 100:.1f}%",
        f"{df['any_chronic'].mean() * 100:.1f}%",
        f"{df['diabetes'].mean() * 100:.1f}%",
        f"{df['hypertension'].mean() * 100:.1f}%",
        f"{df['mental_health'].mean() * 100:.1f}%",
        f"{df['multimorbidity'].mean() * 100:.1f}%",
        f"{df['high_complexity'].mean() * 100:.1f}%",
        f"{df['num_diagnoses'].mean():.2f}",
        f"{df['preventive_visit'].mean() * 100:.1f}%",
        f"{df['is_weekend'].mean() * 100:.1f}%"
    ]
})

print("\n" + summary_stats.to_string(index=False))

# Save summary stats
summary_stats.to_csv('/Users/gihbeom/Desktop/Data Science Preparation/fqhc-chronic-disease-analysis/outputs/summary_statistics.csv', 
                     index=False)
print(f"\n✓ Saved: outputs/summary_statistics.csv")

print("\n" + "="*80)
print("EXPLORATORY DATA ANALYSIS COMPLETE")
print("="*80)
print("✓ Created 4 publication-quality visualizations")
print("✓ Generated summary statistics table")
print("✓ Ready for predictive modeling")